# Your practice dashboard

Run all to refresh. **Not attempted** means no saved attempt; **Attempted** means work was marked or checked; **Passed** means at least one full suite passed. `last_result` still shows a later regression. `checks` counts checker runs, not individual test cases. Use the filters below to pick your next exercise.


In [ ]:
from pathlib import Path
import os
import sys

_candidates = []
if os.environ.get("DE_PRACTICE_ROOT"):
    _candidates.append(Path(os.environ["DE_PRACTICE_ROOT"]))
try:
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _candidates.append(Path("/Workspace") / _nb_path.lstrip("/") )
except Exception:
    pass
_candidates.extend([Path.cwd()] + [Path(p) for p in sys.path if p])
_repo_root = next((parent for candidate in _candidates for parent in [candidate, *candidate.parents]
                   if (parent / "practice_tools" / "engine.py").is_file()
                   and (parent / "exercises" / "37.json").is_file()), None)
if _repo_root is None:
    raise RuntimeError("Open this notebook inside the cloned Git folder. If necessary, set "
                       "DE_PRACTICE_ROOT to that folder's absolute /Workspace/... path.")
sys.path.insert(0, str(_repo_root))
from practice_tools import show_sample, check_solution, mark_attempted, progress_dataframe

if "spark" not in globals():
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.master("local[2]").appName("InterviewPractice").getOrCreate()
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [ ]:
from practice_tools.progress import progress_directory
print("Progress is stored at:", progress_directory(spark))
progress_df = progress_dataframe(spark)
progress_df.groupBy("status").count().show()


In [ ]:
# Optional filters: leave as None to include everything.
status_filter = None  # "Not attempted", "Attempted", or "Passed"
topic_filter = None
difficulty_filter = None  # "Easy", "Medium", or "Hard"
view_df = progress_dataframe(spark)
for name, value in [("status", status_filter), ("topic", topic_filter), ("difficulty", difficulty_filter)]:
    if value is not None:
        view_df = view_df.filter(F.col(name) == value)
view_df = view_df.orderBy(F.col("id").cast("int"))
if "display" in globals():
    display(view_df)
else:
    view_df.show(43, truncate=False)


## Preserve your work

In Databricks, event files are saved outside the Git folder in `/Workspace/Users/<your-user>/.de_interview_practice/events`. Different clones in that workspace share this progress. Locally, they live in `.practice_progress/events` (ignored by Git). These locations do not sync with each other automatically.

Your solution code lives in the notebook: save/commit your notebook changes before pulling conflicting updates or replacing a Git folder. The notebook builder never overwrites existing files.

For a history backup, call `export_progress` with a new absolute filename in your workspace home; it refuses to overwrite an existing backup.
